# Airline Passenger Satisfaction — Exploratory Data Analysis

**Kaggle Playground Series · Classification · Evaluation metric: ROC-AUC**

**Goal:** Inspect the training data, identify basic data-quality issues, and explore how passenger characteristics and service ratings relate to satisfaction. This is exploratory analysis; predictive usefulness will be assessed with validation during modeling.

**Data:** `train.csv` is used for target-related analysis. `test.csv` is reserved for generating competition predictions, with only a brief schema and quality check at the end.

## 1. Load data

In [3]:
from pathlib import Path
import pandas as pd

data_path = Path("../data/raw/train.csv")
df = pd.read_csv(data_path)

## 2. Dataset overview

Check size, column types, and sample rows before making assumptions about the data.

In [5]:
# Shape of the dataset
rows, cols = df.shape
print(f"Dataset contains {rows} rows and {cols} columns")

Dataset contains 699635 rows and 23 columns


In [6]:
# Column names, their dtypes and count of non-null values
print("Dataset contains following columns:")
df.info()

Dataset contains following columns:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 699635 entries, 0 to 699634
Data columns (total 23 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   id                                 699635 non-null  int64  
 1   Age                                699635 non-null  int64  
 2   Flight Distance                    699635 non-null  int64  
 3   Inflight wifi service              699635 non-null  int64  
 4   Departure/Arrival time convenient  699635 non-null  int64  
 5   Ease of Online booking             699635 non-null  int64  
 6   Gate location                      699635 non-null  int64  
 7   Food and drink                     699635 non-null  int64  
 8   Online boarding                    699635 non-null  int64  
 9   Seat comfort                       699635 non-null  int64  
 10  Inflight entertainment             699635 non-null  int64  
 11  On-

In [7]:
# First 5 rows of the dataset
df.head()

,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,...,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,Gender,Customer Type,Type of Travel,Class,satisfaction
0,0,36,694,4,4,4,3,1,4,1,...,4,3,1,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
1,1,56,651,5,5,5,5,5,4,4,...,5,4,3,0,0.0,Male,Loyal Customer,Business travel,Business,True
2,2,31,1371,3,4,2,5,3,3,3,...,4,5,3,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
3,3,10,1616,3,5,3,3,3,3,3,...,3,4,3,0,0.0,Male,Loyal Customer,Personal Travel,Eco,False
4,4,23,481,3,4,3,4,5,3,5,...,5,3,5,5,0.0,Female,disloyal Customer,Business travel,Eco,False


## 3. Target distribution

The target is binary: `True` means satisfied, and `False` means not satisfied. The classes are **55.6% not satisfied** and **44.4% satisfied**, so imbalance is modest. We'll use **ROC-AUC** for model evaluation because that's the competition metric.

In [9]:
target_distribution = (
    df["satisfaction"]
    .value_counts(dropna=False)
    .rename_axis("satisfaction")
    .reset_index(name="count")
)
target_distribution["pct"] = (
    target_distribution["count"] / len(df) * 100
).round(2)
target_distribution

,satisfaction,count,pct
0,False,389296,55.64
1,True,310339,44.36


## 4. Data quality

Check missing values and exact duplicate feature/target rows. The `id` is a unique identifier, not a substantive feature.

In [11]:
# Check for missing values
missing_values_df = df.isna().sum().to_frame()
missing_values_df = missing_values_df.rename(columns={0: "count_of_missing_values"})
missing_values_df["pct"] = missing_values_df["count_of_missing_values"] / rows * 100
missing_values_df

,count_of_missing_values,pct
id,0,0.000000
Age,0,0.000000
Flight Distance,0,0.000000
Inflight wifi service,0,0.000000
Departure/Arrival time convenient,0,0.000000
Ease of Online booking,0,0.000000
Gate location,0,0.000000
Food and drink,0,0.000000
Online boarding,0,0.000000
Seat comfort,0,0.000000


In [12]:
duplicate_count = df.drop(columns="id").duplicated().sum()
print(f"Duplicate rows excluding id: {duplicate_count:,}")

Duplicate rows excluding id: 0


**Finding:** The saved notebook results show **no missing values** and **no duplicate rows** after excluding `id`. This does not rule out all data-quality issues, but it removes two common preprocessing concerns.

## 5. Numeric summary

Use descriptive statistics to spot implausible ranges and understand variable scales. Several service-rating columns contain discrete scores; although stored as integers, they represent **ordinal ratings**, not truly continuous measurements.

In [15]:
df.drop(columns="id").describe().T

,count,mean,std,min,25%,50%,75%,max
Age,699635.0,39.001928,13.800943,7.0,27.0,40.0,50.0,85.0
Flight Distance,699635.0,1353.778702,954.422566,67.0,631.0,954.0,1814.0,4983.0
Inflight wifi service,699635.0,2.778722,1.280419,0.0,2.0,3.0,4.0,5.0
Departure/Arrival time convenient,699635.0,3.156210,1.475390,0.0,2.0,3.0,4.0,5.0
Ease of Online booking,699635.0,2.813240,1.329550,0.0,2.0,3.0,4.0,5.0
Gate location,699635.0,3.013734,1.254014,0.0,2.0,3.0,4.0,5.0
Food and drink,699635.0,3.265644,1.300202,0.0,2.0,3.0,4.0,5.0
Online boarding,699635.0,3.364156,1.288538,0.0,2.0,4.0,4.0,5.0
Seat comfort,699635.0,3.568950,1.275333,0.0,3.0,4.0,5.0,5.0
Inflight entertainment,699635.0,3.453910,1.308157,0.0,2.0,4.0,5.0,5.0


## 6. Categorical distributions

Review the frequency of each category, particularly rare categories that may affect modeling.

In [17]:
# Categorical features: Unique values and frequency counts
categorical_columns = ["Gender", "Customer Type", "Type of Travel", "Class"]
for col in categorical_columns:
    print(f"Unique values and frequency of column {col}:")
    print(df[col].value_counts())
    print()

Unique values and frequency of column Gender:
Gender
Male      351683
Female    347952
Name: count, dtype: int64

Unique values and frequency of column Customer Type:
Customer Type
Loyal Customer       576990
disloyal Customer    122645
Name: count, dtype: int64

Unique values and frequency of column Type of Travel:
Type of Travel
Business travel    497441
Personal Travel    202194
Name: count, dtype: int64

Unique values and frequency of column Class:
Class
Business    342212
Eco         327404
Eco Plus     30019
Name: count, dtype: int64



**Observations:** Gender is approximately balanced. Loyal customers account for roughly **82.5%** of passengers, business travelers roughly **71.1%**, and `Eco Plus` accounts for only around **4.3%** of records.

## 7. Numerical features vs. satisfaction

Encode the boolean target as 0/1 and calculate **Pearson correlations**. Rank by *absolute* magnitude but retain the sign to show direction. Correlation measures linear association, not causation or feature importance. Service ratings are ordinal, so treat these Pearson results as an initial descriptive screen.

In [20]:
# Keep the original boolean target in df; create a numeric view for correlation.
target_numeric = df["satisfaction"].astype(int)

numeric_features = df.select_dtypes(include="number").columns.drop("id", errors="ignore")
correlations = df[numeric_features].corrwith(target_numeric)
correlation_table = (
    correlations.rename("pearson_r")
    .to_frame()
    .assign(abs_r=lambda x: x["pearson_r"].abs())
    .sort_values("abs_r", ascending=False)
)
correlation_table

,pearson_r,abs_r
Online boarding,0.559583,0.559583
Inflight entertainment,0.430755,0.430755
Seat comfort,0.401488,0.401488
Flight Distance,0.370038,0.370038
On-board service,0.346618,0.346618
Cleanliness,0.338853,0.338853
Leg room service,0.330116,0.330116
Inflight wifi service,0.291275,0.291275
Baggage handling,0.258118,0.258118
Checkin service,0.249799,0.249799


**Top five by absolute Pearson correlation (from the saved results):** `Online boarding` (0.560), `Inflight entertainment` (0.431), `Seat comfort` (0.401), `Flight Distance` (0.370), and `On-board service` (0.347). These previously displayed values were absolute; the updated table will also show the **direction** when run.

**Interpretation:** The service ratings and flight distance deserve attention during modeling. Low Pearson correlations for delays do **not** mean delays are necessarily useless predictors, since relationships may be nonlinear or conditional on other features.

## 8. Categorical features vs. satisfaction

For each category, calculate the proportion of passengers satisfied. Compare **rates**, not raw counts, because group sizes differ.

In [23]:
categorical_columns = ["Gender", "Customer Type", "Type of Travel", "Class"]

for col in categorical_columns:
    satisfaction_rates = (
        df.groupby(col)["satisfaction"].mean().mul(100).sort_values(ascending=False)
    )
    print(f"\n{col} — satisfied (%)")
    print(satisfaction_rates.round(1).to_string())


Gender — satisfied (%)
Gender
Male      44.9
Female    43.8

Customer Type — satisfied (%)
Customer Type
Loyal Customer       49.5
disloyal Customer    20.1

Type of Travel — satisfied (%)
Type of Travel
Business travel    58.4
Personal Travel     9.7

Class — satisfied (%)
Class
Business    72.5
Eco Plus    24.2
Eco         16.8


**Findings from the saved results:**

- **Class:** Business 72.5%, Eco Plus 24.2%, Eco 16.8% satisfied.
- **Type of Travel:** Business travel 58.4% vs personal travel 9.7%.
- **Customer Type:** Loyal 49.5% vs disloyal 20.1%.
- **Gender:** Male 44.9% vs female 43.8%, a small difference.

These are **unadjusted associations**: for example, travel class and travel type might overlap. They do not establish causal effects.

## 9. Quick test-set consistency check

The Kaggle test set has no satisfaction labels, so do **not** use it to calculate target relationships or validate the model. Check its columns, missingness, and broad category compatibility only.

In [26]:
test_path = data_path.with_name("test.csv")
test_df = pd.read_csv(test_path)

train_features = set(df.columns) - {"satisfaction"}
test_features = set(test_df.columns)

print(f"Train shape: {df.shape}; test shape: {test_df.shape}")
print("Train-only feature columns:", sorted(train_features - test_features))
print("Test-only columns:", sorted(test_features - train_features))
print("Missing values in test:", int(test_df.isna().sum().sum()))

for col in categorical_columns:
    unexpected = set(test_df[col].dropna().unique()) - set(df[col].dropna().unique())
    print(f"Unseen test categories in {col}: {sorted(unexpected)}")

Train shape: (699635, 23); test shape: (299844, 22)
Train-only feature columns: []
Test-only columns: []
Missing values in test: 130
Unseen test categories in Gender: []
Unseen test categories in Customer Type: []
Unseen test categories in Type of Travel: []
Unseen test categories in Class: []


## 10. Conclusions and next steps

- The training data appears complete with no exact duplicates (per the saved checks).
- The target is reasonably balanced, with ~44.4% satisfied passengers.
- Service ratings, **Class**, and **Type of Travel** show substantial univariate associations with satisfaction.
- **Gender** has a small univariate association; don't drop it just from this observation.
- Next: build a reproducible preprocessing pipeline, train a baseline classifier, compare ROC-AUC on held-out training data, and generate predictions for the competition test set.

**Important:** No features are removed purely because of low univariate correlation. Actual feature selection should be informed by validation results.